# From `System` to compiled dynamics primitives

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/alx87grd/minilink/blob/main/examples/notebooks/intro/07_compile.ipynb)

Minilink models are **textbook `System` objects** — you write `f(x, u, t, params)` and
output maps once. **Compilation** lowers a leaf or wired diagram into a flat
**evaluator**: a collection of fast dynamic primitives (`f`, `rk4_step`,
`rk4_integrate_zoh`, …) backed by NumPy or JAX.

This notebook is a focused introduction to that pipeline:

1. [The model stays textbook](#1.-The-model-stays-textbook)
2. [`compile(backend)` → evaluator](#2.-compilebackend--evaluator)
3. [What compilation does to a diagram](#3.-What-compilation-does-to-a-diagram)
4. [Fast tier: dynamics primitives](#4.-Fast-tier-dynamics-primitives)
5. [NumPy vs JAX backends](#5.-NumPy-vs-JAX-backends)
6. [Trace tier (brief)](#6.-Trace-tier-brief)
7. [Where evaluators plug in](#7.-Where-evaluators-plug-in)

**Prerequisites:** basic minilink `System` / diagram usage
([showcase/minilink.ipynb](../showcase/minilink.ipynb)). For the *why* of stateless `f` and
JAX tracing, see [showcase/jax.ipynb](../showcase/jax.ipynb).

**Run locally** with minilink installed (`pip install -e .` from the repo root), or
open on Colab — the setup cell detects Colab and clones the repo automatically.


In [ ]:
# Local conda: minilink already installed. Colab: clone + path + meshcat.
import sys

if "google.colab" in sys.modules:
    get_ipython().run_line_magic("matplotlib", "inline")
    get_ipython().system("git clone https://github.com/alx87grd/minilink")
    sys.path.insert(0, "/content/minilink")
    get_ipython().system("pip install -q meshcat")


## 1. The model stays textbook

A continuous-time plant implements one dynamics equation plus boundary outputs:

    dx = f(x, u, t, params)
    y  = h(x, u, t, params)   # one map per output port

Simulation facades (`compute_trajectory`, …) call **compile internally**, but you
can also compile once and hold the evaluator when you need repeated fast calls
(MPC inner loops, benchmarks, custom integrators, NLP defects).


In [ ]:
import numpy as np

from minilink.dynamics.catalog.pendulum.pendulum import Pendulum

plant = Pendulum()
plant.params["d"] = 0.2
plant.x0 = np.array([0.5, 0.0])

x = plant.x0.copy()
u = np.array([0.0])

print("plant.f(x, u) =", plant.f(x, u))
plant

## 2. `compile(backend)` → evaluator

`sys.compile(backend="numpy" | "jax")` returns a **typed evaluator**:

| Model kind | Evaluator | Primary primitives |
| --- | --- | --- |
| `DynamicSystem` leaf or `DiagramSystem` | `DynamicsEvaluator` | `.f`, integration steps/rollouts, `.outputs` |
| `StepSystem` / step diagram | `StepEvaluator` | `.step`, state-only `.rollout`, `.outputs` |
| static leaf (`n=0`) | `StaticEvaluator` | `.outputs` only |

Default methods (`.f`, `.rk4_step`, …) are the **fast tier** — eager NumPy or
JIT-compiled JAX. JAX evaluators also expose a **trace tier** (`.f_trace`, …) for
composition inside outer `jit` / `grad` (Section 6).


In [ ]:
np_eval = plant.compile(backend="numpy")
jax_eval = plant.compile(backend="jax")

print(type(np_eval).__name__, "backend =", np_eval.backend)
print(type(jax_eval).__name__, "backend =", jax_eval.backend)
print("JAX trace tier available:", jax_eval.has_trace_tier)
print("state dim n =", jax_eval.n, ", input dim m =", jax_eval.m)

dx_np = np_eval.f(x, u)
dx_jax = np.asarray(jax_eval.f(x, u))
print("np_eval.f  =", dx_np)
print("jax_eval.f =", dx_jax)

## 3. What compilation does to a diagram

On a **wired diagram**, compilation is a two-stage lowering:

1. **Topology pass** — resolve port dependencies, detect algebraic loops, produce
   a fixed evaluation order.
2. **`ExecutionPlan`** — flatten subsystem `f` / port computes into one state
   vector and one internal signal buffer; wire gathers with `gather_u`.

The evaluator then exposes the **same primitive API** as a leaf: one call to `.f`
runs the whole diagram without recursive Python dispatch.


In [ ]:
from minilink.blocks.sources import Step
from minilink.control.impedance import ImpedanceController

step = Step()
step.params["final_value"] = np.array([1.0])
step.params["step_time"] = 2.0

ctl = ImpedanceController()
ctl.params["Kp"] = 50.0
ctl.params["Kd"] = 5.0

diagram = step >> ctl @ plant
diagram

In [ ]:
diag_eval = diagram.compile(backend="jax", verbose=True)

print("plan state_dim:", diag_eval.plan.state_dim)
print("plan signal_dim:", diag_eval.plan.signal_dim)
print("port ops:", len(diag_eval.plan.port_ops))
print("state ops:", len(diag_eval.plan.state_ops))

x_diag = np.zeros(diagram.n)
u_diag = diagram.get_u_from_input_ports()
print("diagram.f(x, u) =", np.asarray(diag_eval.f(x_diag, u_diag)))

## 4. Fast tier: dynamics primitives

A compiled **dynamics evaluator** is a toolbox of reusable numerical kernels.
Think in layers:

| Layer | Examples | Role |
| --- | --- | --- |
| **Dynamics** | `f`, `f_p`, `f_ivp`, `f_ivp_p` | One evaluation of $\dot x$ |
| **Outputs** | `outputs`, `outputs_p` | Boundary port dict (not diagram internals) |
| **Single step** | `rk4_step`, `euler_step`, `*_ivp` variants | Advance one $\Delta t$ |
| **Rollouts** | `rk4_integrate_zoh`, `rk4_integrate_linear`, `euler_integrate_zoh`, … | Many steps in one call |
| **Bridges** | `f_scipy`, `as_scipy_rhs`, `integrate_zoh_rollout` | Solver / hybrid adapters |

**Naming grammar** (rollouts always state the input model):

    {rk4|euler}_{step|integrate}_{zoh|linear|ivp}

Single-step names omit `_zoh` — holding `u` over $\Delta t$ is the default.
Parametric twins append `_p`; JAX trace twins append `_trace` / `_trace_p`.


In [ ]:
dt = 0.01
t0 = 0.0

# --- dynamics + outputs ---
dx = jax_eval.f(x, u, t0)
y = jax_eval.outputs(x, u, t0)
print("f(x,u)     =", np.asarray(dx))
print("outputs    =", {k: np.asarray(v) for k, v in y.items()})

# --- single RK4 step (ZOH on u) ---
x1 = np.asarray(jax_eval.rk4_step(x, u, t0, dt))
print("rk4_step   =", x1)

# --- rollout: hold u over a uniform grid ---
n_steps = 50
u_seq = np.tile(u, (n_steps, 1))
x_path = np.asarray(jax_eval.rk4_integrate_zoh(x, u_seq, t0, dt))
print("rk4_integrate_zoh shape:", x_path.shape, "  final x =", x_path[-1])

Sugar helpers wrap the same primitives:

- `integrate_zoh` — one hold window (optional inner substeps)
- `integrate_zoh_rollout` — returns `(t_samples, x_samples)` for plotting

Discrete **step** evaluators expose `.step` and a state-only `.rollout(k, x, u)`.
Signal histories for scheduled diagrams live in `Computer` / `HybridSimulator`, not
in the evaluator rollout.


In [ ]:
t_samples, x_samples = jax_eval.integrate_zoh_rollout(
    x, u_hold=u, t0=t0, dt_hold=n_steps * dt, dt_inner=dt
)
print("integrate_zoh_rollout:", t_samples.shape, x_samples.shape)

## 5. NumPy vs JAX backends

Both backends expose the **same public API**. The difference is execution:

- **NumPy** — eager Python loops for rollouts; good for debugging and small models.
- **JAX** — `.f` and rollouts JIT to XLA; first call pays compile cost, then fast.

On a diagram, the win is eliminating recursive port dispatch — plus JAX fusion
when the backend is `"jax"`.


In [ ]:
import time

n = 2000
xc = np.zeros(plant.n)
uc = np.zeros(plant.m)

for label, fn in [
    ("plant.f (recursive)", lambda: plant.f(xc, uc)),
    ("NumPy evaluator.f", lambda: np_eval.f(xc, uc)),
    ("JAX evaluator.f", lambda: jax_eval.f(xc, uc)),
]:
    out = fn()
    if hasattr(out, "block_until_ready"):
        out.block_until_ready()
    t0_b = time.perf_counter()
    for _ in range(n):
        out = fn()
        if hasattr(out, "block_until_ready"):
            out.block_until_ready()
    dt_b = time.perf_counter() - t0_b
    print(f"{label:22s}: {1e6 * dt_b / n:6.1f} µs/call")

## 6. Trace tier (brief)

Fast-tier `.f` is already JIT-compiled on JAX. The **trace tier** (`.f_trace`,
`.f_trace_p`, `.rk4_step_trace`, …) returns **pre-JIT flat callables** so an
*outer* `jax.jit(jax.grad(...))` can differentiate through them.

NumPy evaluators reject `*_trace` / `*_jit` attributes — use JAX when you need AD
through compiled dynamics.

See [../showcase/jax.ipynb](../showcase/jax.ipynb) for
the full story on stateless `f`, tracing, and parameter Jacobians.


In [ ]:
import jax
import jax.numpy as jnp

from minilink.core.backends import configure_jax

configure_jax(enable_x64=True)

x_j = jnp.array(x)
u_j = jnp.array(u)
params = {"m": 1.0, "l": 1.0, "I": 1.0, "gravity": 9.81, "d": 0.2}

A = jax.jacfwd(lambda xx: jax_eval.f_trace(xx, u_j, 0.0))(x_j)
print("A = df/dx (trace tier):\n", np.asarray(A).round(3))

d_d = jax.grad(
    lambda d_val: jnp.sum(
        jax_eval.f_trace_p(
            x_j, u_j, 0.0, {"m": 1.0, "l": 1.0, "I": 1.0, "gravity": 9.81, "d": d_val}
        )
        ** 2
    )
)(jnp.array(0.2))
print("d(loss)/d(damping) =", float(d_d))

## 7. Where evaluators plug in

Compiled evaluators are the **fast math layer** behind higher-level workflows:

| Consumer | Typical evaluator use |
| --- | --- |
| `Simulator` | `compile(backend=...)` then integrate (`rk4`, `euler`, `scipy_ivp`, …) |
| `compute_trajectory` façade | compile + solve; caches `Trajectory` |
| Trajectory optimization / MPC | JAX `f_trace` / rollouts for defects and gradients |
| `HybridSimulator` | plant `integrate_zoh_rollout`; computer uses its own tick engine |
| Custom scripts | hold `evaluator = sys.compile(...)` and call primitives directly |

**Mental model:** write equations on `System` objects → **compile once** → call
primitives many times. Facades and simulators are thin orchestration on top.

### See also

- [`showcase/minilink.ipynb`](../showcase/minilink.ipynb) — product tour
- [`showcase/jax.ipynb`](../showcase/jax.ipynb) — why pure `f` enables AD (marketing)
- [`tooling/benchmark.ipynb`](../tooling/benchmark.ipynb) — solver × backend sweeps
- Design contracts: `DESIGN.md` §5
